# 03 — Diagnóstico de datos faltantes en `vflow` y estrategia de imputación

Este notebook evalúa la cobertura de `gtmd2_vflow_int` para los **10 países de Sudamérica** (excluyendo GUY/SUR/GUF) y recomienda una estrategia de imputación para cada corredor con datos faltantes.

**Fuentes alternativas evaluadas:**

| Fuente | Cobertura | Corredores |
|---|---|---|
| `turismo_anio_ARG.csv` | 1990–2022 | BOL/BRA/CHL/PRY/URY → ARG |
| `tourism_UN_ARG_Latam_clean.csv` | 1995–2022 (parcial) | COL/ECU/PER/VEN → ARG (desde 2016) |
| UNWTO (columnas GTMD2) | variable | todos los demás corredores |

**Técnicas evaluadas:**
- **A) Directo** — usar fuente alternativa sin ajuste (overlap insuficiente para ratio)
- **B) Rescalar** — ratio corredor-específico en años con ambas series (≥ 3 años overlap)
- **C) Interpolación log-lineal** — gap rodeado de datos, sin fuente alternativa
- **D) Sin imputación** — sin datos adyacentes ni fuente alternativa

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)
pd.set_option('display.max_rows', 120)

In [ ]:
# ── Rutas ─────────────────────────────────────────────────────────────────────
GTMD2_PATH    = 'C:/Data/GTMD2/GTMD2_Data_MIGMOBS_share.csv'
ARG_TOUR_PATH = 'Data/turismo_anio_ARG.csv'
ARG_UN_PATH   = 'Data/tourism_UN_ARG_Latam_clean.csv'

# ── Países ────────────────────────────────────────────────────────────────────
SA10 = ['ARG', 'BOL', 'BRA', 'CHL', 'COL', 'ECU', 'PRY', 'PER', 'URY', 'VEN']
YEAR_MIN, YEAR_MAX = 1995, 2022
ALL_YEARS = list(range(YEAR_MIN, YEAR_MAX + 1))
N_YEARS   = len(ALL_YEARS)

GTMD2_COLS = [
    'iso3code_i', 'iso3code_j', 'year', 'country_i', 'country_j',
    'gtmd2_vflow_int',
    'unwto_tflow_112', 'unwto_tflow_111', 'unwto_tflow_122', 'unwto_tflow_121',
]
UNWTO_COLS = ['unwto_tflow_112', 'unwto_tflow_111', 'unwto_tflow_122', 'unwto_tflow_121']

# Orígenes cubiertos por turismo_anio_ARG (INDEC)
ARG_TOUR_MAP = {'Bolivia': 'BOL', 'Brasil': 'BRA', 'Chile': 'CHL',
                'Paraguay': 'PRY', 'Uruguay': 'URY'}
MIN_OVERLAP  = 3   # años mínimos de overlap para ratio corredor-específico

---
## Sección 1 — Carga de datos

In [ ]:
print('Cargando GTMD2...')
gtmd2_raw = pd.read_csv(GTMD2_PATH, usecols=GTMD2_COLS, low_memory=False)
gtmd2 = gtmd2_raw[
    gtmd2_raw['iso3code_i'].isin(SA10) &
    gtmd2_raw['iso3code_j'].isin(SA10) &
    (gtmd2_raw['iso3code_i'] != gtmd2_raw['iso3code_j']) &
    gtmd2_raw['year'].between(YEAR_MIN, YEAR_MAX)
].copy()
del gtmd2_raw
print(f'  {len(gtmd2):,} filas  |  {gtmd2.groupby(["iso3code_i","iso3code_j"]).ngroups} corredores')
print(f'  gtmd2_vflow_int no-NaN: {gtmd2["gtmd2_vflow_int"].notna().sum():,} ({100*gtmd2["gtmd2_vflow_int"].notna().mean():.1f}%)')

In [ ]:
# ── turismo_anio_ARG (INDEC) — BOL/BRA/CHL/PRY/URY → ARG ─────────────────────
arg_tour = (
    pd.read_csv(ARG_TOUR_PATH)
    .query('pais_agrupado in @ARG_TOUR_MAP')
    .assign(iso3code_i=lambda x: x['pais_agrupado'].map(ARG_TOUR_MAP),
            iso3code_j='ARG',
            alt_value=lambda x: x['Viajes'],
            alt_source='turismo_anio_ARG')
    .rename(columns={'anio': 'year'})
    [['iso3code_i', 'iso3code_j', 'year', 'alt_value', 'alt_source']]
    .query(f'{YEAR_MIN} <= year <= {YEAR_MAX}')
)

# ── tourism_UN_ARG_Latam — resto de países → ARG ──────────────────────────────
arg_un = (
    pd.read_csv(ARG_UN_PATH)
    .rename(columns={'un_latam_trips': 'alt_value'})
    .assign(iso3code_j='ARG', alt_source='tourism_UN_ARG_Latam')
    [['iso3code_i', 'iso3code_j', 'year', 'alt_value', 'alt_source']]
    .query(f'{YEAR_MIN} <= year <= {YEAR_MAX}')
)

# Combinar: turismo_anio tiene precedencia para BOL/BRA/CHL/PRY/URY
arg_alt = pd.concat([
    arg_tour,
    arg_un[~arg_un['iso3code_i'].isin(list(ARG_TOUR_MAP.values()))]
], ignore_index=True)
arg_alt_lookup = arg_alt.set_index(['iso3code_i', 'year'])

print('Fuentes alternativas para j=ARG:')
print(arg_alt.groupby(['iso3code_i', 'alt_source']).agg(
    anios=('year', 'count'),
    primer_anio=('year', 'min'),
    ultimo_anio=('year', 'max'),
    non_nan=('alt_value', 'count')
).to_string())

---
## Sección 2 — Mapa de cobertura de `gtmd2_vflow_int`

Cada celda muestra si el corredor (i → j) tiene dato en ese año.
- **Verde oscuro** = dato disponible
- **Rojo** = NaN

In [ ]:

# ── Construir matriz de técnica por corredor × año ────────────────────────────
# Primero necesitamos correr la evaluación (se repite abajo, pero aquí
# construimos la matriz visual antes de mostrar la tabla detallada)

# Pivot cobertura base
cov_matrix = (
    gtmd2
    .assign(has_data=gtmd2['gtmd2_vflow_int'].notna().astype(int))
    .pivot_table(index=['iso3code_i', 'iso3code_j'], columns='year',
                 values='has_data', aggfunc='max')
    .reindex(columns=ALL_YEARS)
    .fillna(0)
    .astype(int)
)
# Añadir corredores completamente ausentes
for i in SA10:
    for j in SA10:
        if i != j and (i, j) not in cov_matrix.index:
            cov_matrix.loc[(i, j), :] = 0
cov_matrix = cov_matrix.sort_index()

# ── Heatmap simple: dato / no dato ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 11))
cmap = mcolors.ListedColormap(['#f8d7da', '#2ca02c'])   # rojo=NaN, verde=dato
im = ax.imshow(cov_matrix.values, aspect='auto', cmap=cmap,
               vmin=0, vmax=1, interpolation='nearest')

ax.set_xticks(range(N_YEARS))
ax.set_xticklabels(ALL_YEARS, rotation=90, fontsize=8)
ax.set_yticks(range(len(cov_matrix)))
ax.set_yticklabels([f'{i} → {j}' for i, j in cov_matrix.index], fontsize=7)
ax.set_title('Cobertura de gtmd2_vflow_int — SA10\nVerde = dato  |  Rojo = NaN', fontsize=11)
plt.tight_layout()
plt.show()


---
## Sección 3 — Evaluación corredor por corredor

Para cada corredor con faltantes se evalúa:
1. Qué años faltan
2. Qué fuente alternativa cubre esos años (UNWTO o fuentes ARG)
3. Si hay overlap suficiente para calcular un ratio de rescalado
4. Si el gap es interpolable (datos en ambos extremos)

In [ ]:
def best_unwto(row):
    for col in UNWTO_COLS:
        v = row.get(col, np.nan)
        if pd.notna(v):
            return v, col
    return np.nan, None


def evaluate_corridor(sub, iso_i, iso_j):
    sub = sub.set_index('year').reindex(ALL_YEARS)
    gtmd2_vals = sub['gtmd2_vflow_int']
    n_gtmd2    = int(gtmd2_vals.notna().sum())
    missing_yrs = gtmd2_vals[gtmd2_vals.isna()].index.tolist()

    if not missing_yrs:
        return dict(iso3code_i=iso_i, iso3code_j=iso_j,
                    n_gtmd2=n_gtmd2, n_missing=0, missing_years=[],
                    tecnica='sin faltantes', fuente_alt='', notas='')

    years_with_data = sorted(gtmd2_vals.dropna().index.tolist())

    # ── UNWTO en años faltantes ───────────────────────────────────────────────
    n_unwto_missing = sum(
        1 for yr in missing_yrs
        if yr in sub.index and pd.notna(best_unwto(sub.loc[yr])[0])
    )
    # Overlap gtmd2 ∩ UNWTO (para calcular ratio)
    n_overlap_unwto = sum(
        1 for yr in ALL_YEARS
        if yr in sub.index
        and pd.notna(sub.loc[yr, 'gtmd2_vflow_int'])
        and pd.notna(best_unwto(sub.loc[yr])[0])
    )

    # ── Fuentes ARG ───────────────────────────────────────────────────────────
    n_alt_arg, n_overlap_arg, alt_sources_arg = 0, 0, set()
    if iso_j == 'ARG':
        for yr in missing_yrs:
            key = (iso_i, yr)
            if key in arg_alt_lookup.index:
                row_alt = arg_alt_lookup.loc[key]
                val = row_alt['alt_value'] if isinstance(row_alt, pd.Series) else row_alt['alt_value'].iloc[0]
                src = row_alt['alt_source'] if isinstance(row_alt, pd.Series) else row_alt['alt_source'].iloc[0]
                if pd.notna(val):
                    n_alt_arg += 1
                    alt_sources_arg.add(src)
        # Overlap gtmd2 ∩ fuente ARG
        for yr in years_with_data:
            key = (iso_i, yr)
            if key in arg_alt_lookup.index:
                row_alt = arg_alt_lookup.loc[key]
                val = row_alt['alt_value'] if isinstance(row_alt, pd.Series) else row_alt['alt_value'].iloc[0]
                if pd.notna(val):
                    n_overlap_arg += 1

    # ── Interpolabilidad ─────────────────────────────────────────────────────
    n_interpolable = 0
    if years_with_data:
        first_d, last_d = min(years_with_data), max(years_with_data)
        n_interpolable = sum(1 for yr in missing_yrs if first_d < yr < last_d)

    # ── Decisión ─────────────────────────────────────────────────────────────
    notas = []

    if iso_j == 'ARG' and n_alt_arg > 0:
        fuente = ' + '.join(sorted(alt_sources_arg))
        if n_overlap_arg >= MIN_OVERLAP:
            tecnica = 'B) Rescalar fuente ARG'
            notas.append(f'{n_overlap_arg} años overlap para ratio')
        else:
            tecnica = 'A) Usar fuente ARG directo'
            notas.append(f'solo {n_overlap_arg} años overlap')
        if n_alt_arg < len(missing_yrs):
            resto = len(missing_yrs) - n_alt_arg
            notas.append(f'fuente ARG cubre {n_alt_arg}/{len(missing_yrs)} años faltantes')
            if n_interpolable > 0:
                notas.append(f'{n_interpolable} años restantes interpolables')
            else:
                notas.append(f'{resto} años sin cobertura posible')

    elif n_unwto_missing > 0:
        fuente = 'UNWTO'
        if n_overlap_unwto >= MIN_OVERLAP:
            tecnica = 'B) Rescalar UNWTO'
            notas.append(f'{n_overlap_unwto} años overlap para ratio')
        else:
            tecnica = 'A) Usar UNWTO directo'
            notas.append(f'solo {n_overlap_unwto} años overlap')
        if n_unwto_missing < len(missing_yrs):
            notas.append(f'UNWTO cubre {n_unwto_missing}/{len(missing_yrs)} años faltantes')
            if n_interpolable > 0:
                notas.append(f'{n_interpolable} años restantes interpolables')

    elif n_interpolable > 0:
        tecnica = 'C) Interpolacion log-lineal'
        fuente  = '(datos adyacentes)'
        n_extremos = len(missing_yrs) - n_interpolable
        notas.append(f'{n_interpolable}/{len(missing_yrs)} interpolables')
        if n_extremos > 0:
            notas.append(f'{n_extremos} en extremos sin cobertura')

    else:
        tecnica = 'D) Sin imputacion posible'
        fuente  = ''
        notas.append('sin UNWTO, sin fuente ARG, sin datos adyacentes')

    return dict(
        iso3code_i=iso_i, iso3code_j=iso_j,
        n_gtmd2=n_gtmd2, n_missing=len(missing_yrs),
        missing_years=missing_yrs,
        n_unwto_en_faltantes=n_unwto_missing,
        n_overlap_rescale=n_overlap_arg if (iso_j == 'ARG' and n_alt_arg > 0) else n_overlap_unwto,
        n_interpolable=n_interpolable,
        tecnica=tecnica, fuente_alt=fuente,
        notas=' | '.join(notas),
    )

In [ ]:
results = []
existing_pairs = set(gtmd2.groupby(['iso3code_i', 'iso3code_j']).groups.keys())

for (iso_i, iso_j), sub in gtmd2.groupby(['iso3code_i', 'iso3code_j']):
    results.append(evaluate_corridor(sub.copy(), iso_i, iso_j))

# Corredores completamente ausentes del GTMD2
for iso_i in SA10:
    for iso_j in SA10:
        if iso_i != iso_j and (iso_i, iso_j) not in existing_pairs:
            results.append(dict(
                iso3code_i=iso_i, iso3code_j=iso_j,
                n_gtmd2=0, n_missing=N_YEARS,
                missing_years=ALL_YEARS,
                n_unwto_en_faltantes=0, n_overlap_rescale=0, n_interpolable=0,
                tecnica='D) Sin imputacion posible',
                fuente_alt='', notas='corredor AUSENTE en GTMD2'
            ))

diag = pd.DataFrame(results).sort_values(['iso3code_j', 'iso3code_i']).reset_index(drop=True)

print(f'Total corredores evaluados : {len(diag)}')
print(f'Sin faltantes              : {(diag["n_missing"] == 0).sum()}')
print(f'Con faltantes              : {(diag["n_missing"] > 0).sum()}')

In [ ]:

# ── Listados por categoría ────────────────────────────────────────────────────
interpolables  = diag[diag['tecnica'] == 'C) Interpolacion log-lineal']
sin_imputacion = diag[diag['tecnica'] == 'D) Sin imputacion posible']

print(f"{'='*55}")
print(f"  C) INTERPOLACIÓN LOG-LINEAL  ({len(interpolables)} corredores)")
print(f"{'='*55}")
for _, r in interpolables.sort_values(['iso3code_j', 'iso3code_i']).iterrows():
    print(f"  {r['iso3code_i']:3s} → {r['iso3code_j']:3s}  |  {r['n_missing']:2d} años faltantes  |  {r['notas']}")

print(f"\n{'='*55}")
print(f"  D) SIN IMPUTACIÓN POSIBLE  ({len(sin_imputacion)} corredores)")
print(f"{'='*55}")
for _, r in sin_imputacion.sort_values(['iso3code_j', 'iso3code_i']).iterrows():
    print(f"  {r['iso3code_i']:3s} → {r['iso3code_j']:3s}  |  {r['n_missing']:2d} años faltantes  |  {r['notas']}")

# ── Heatmap coloreado por técnica ─────────────────────────────────────────────
# Código numérico por técnica (para celdas SIN dato)
# 0=D(sin imputación), 1=C(interpolable), 2=A(directo), 3=B(rescalar), 4=dato
TECNICA_MISSING_CODE = {
    'D) Sin imputacion posible'  : 0,
    'C) Interpolacion log-lineal': 1,
    'A) Usar fuente ARG directo' : 2,
    'A) Usar UNWTO directo'      : 2,
    'B) Rescalar fuente ARG'     : 3,
    'B) Rescalar UNWTO'          : 3,
}

# Construir plot_matrix directamente
plot_matrix = pd.DataFrame(0, index=cov_matrix.index, columns=cov_matrix.columns)

# Celdas con dato → código 4
plot_matrix[cov_matrix == 1] = 4

# Celdas sin dato → código según técnica del corredor
for _, row in diag.iterrows():
    iso_i, iso_j = row['iso3code_i'], row['iso3code_j']
    tecnica = row['tecnica']
    if (iso_i, iso_j) not in plot_matrix.index:
        continue
    code = TECNICA_MISSING_CODE.get(tecnica)
    if code is None or not row['missing_years']:
        continue
    missing_yrs = [yr for yr in row['missing_years'] if yr in ALL_YEARS]
    for yr in missing_yrs:
        if yr in plot_matrix.columns:
            plot_matrix.loc[(iso_i, iso_j), yr] = code

# ── Colores y leyenda ─────────────────────────────────────────────────────────
COLORS = ['#d62728', '#ff7f0e', '#fff3cd', '#cce5ff', '#2ca02c']
LABELS = ['D) Sin imputación', 'C) Interpolable', 'A) Usar directo', 'B) Rescalar', 'Dato disponible']

cmap5  = mcolors.ListedColormap(COLORS)
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]
norm   = mcolors.BoundaryNorm(bounds, cmap5.N)

fig, ax = plt.subplots(figsize=(18, 11))
ax.imshow(plot_matrix.values, aspect='auto', cmap=cmap5, norm=norm,
          interpolation='nearest')
ax.set_xticks(range(N_YEARS))
ax.set_xticklabels(ALL_YEARS, rotation=90, fontsize=8)
ax.set_yticks(range(len(plot_matrix)))
ax.set_yticklabels([f'{i} → {j}' for i, j in plot_matrix.index], fontsize=7)
ax.set_title('Estrategia de imputación por corredor × año — SA10', fontsize=12, pad=12)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for c, l in zip(COLORS, LABELS)]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9,
          bbox_to_anchor=(1.15, 1), borderaxespad=0)
plt.tight_layout()
plt.show()


---
## Sección 4 — Tabla resumen: corredores con faltantes

In [ ]:
# Paleta de colores por técnica
COLOR_MAP = {
    'sin faltantes'             : '#d4edda',  # verde claro
    'B) Rescalar fuente ARG'    : '#cce5ff',  # azul claro
    'B) Rescalar UNWTO'         : '#cce5ff',
    'A) Usar fuente ARG directo': '#fff3cd',  # amarillo
    'A) Usar UNWTO directo'     : '#fff3cd',
    'C) Interpolacion log-lineal': '#fce8d0', # naranja claro
    'D) Sin imputacion posible' : '#f8d7da',  # rojo claro
}

def color_row(row):
    color = COLOR_MAP.get(row['tecnica'], '#ffffff')
    return [f'background-color: {color}'] * len(row)

con_faltantes = diag[diag['n_missing'] > 0].copy()

cols_show = ['iso3code_i', 'iso3code_j', 'n_gtmd2', 'n_missing',
             'n_unwto_en_faltantes', 'n_overlap_rescale', 'n_interpolable',
             'tecnica', 'fuente_alt', 'notas']

display(
    con_faltantes[cols_show]
    .style
    .apply(color_row, axis=1)
    .set_caption(f'Corredores con datos faltantes en gtmd2_vflow_int ({len(con_faltantes)} de {len(diag)})')
    .set_properties(**{'font-size': '11px'})
)

print('\nDistribucion por tecnica:')
print(diag['tecnica'].value_counts().to_string())

---
## Sección 5 — Detalle por país destino

Para cada destino con problemas: qué años faltan exactamente y qué valores alternativos hay disponibles.

In [ ]:
for iso_j in SA10:
    problemas = diag[(diag['iso3code_j'] == iso_j) & (diag['n_missing'] > 0)]
    if len(problemas) == 0:
        continue

    print(f'\n{"="*60}')
    print(f'  DESTINO: {iso_j}  ({len(problemas)} corredores con faltantes)')
    print(f'{"="*60}')

    for _, row in problemas.iterrows():
        iso_i = row['iso3code_i']
        missing_yrs = row['missing_years']
        print(f'\n  {iso_i} -> {iso_j}  |  {row["n_missing"]} años faltantes  |  {row["tecnica"]}')
        if row['notas']:
            print(f'  Notas: {row["notas"]}')
        print(f'  Anos faltantes: {missing_yrs}')

        # Mostrar valores alternativos disponibles para esos años
        sub = gtmd2[
            (gtmd2['iso3code_i'] == iso_i) &
            (gtmd2['iso3code_j'] == iso_j) &
            gtmd2['year'].isin(missing_yrs)
        ][['year', 'gtmd2_vflow_int'] + UNWTO_COLS].set_index('year').reindex(missing_yrs)

        if iso_j == 'ARG':
            alt_vals = {
                yr: arg_alt_lookup.loc[(iso_i, yr)] if (iso_i, yr) in arg_alt_lookup.index else None
                for yr in missing_yrs
            }
            sub['fuente_ARG'] = [
                (v['alt_value'] if isinstance(v, pd.Series) else v['alt_value'].iloc[0])
                if v is not None else np.nan
                for v in alt_vals.values()
            ]
            sub['fuente_ARG_src'] = [
                (v['alt_source'] if isinstance(v, pd.Series) else v['alt_source'].iloc[0])
                if v is not None else ''
                for v in alt_vals.values()
            ]

        display(sub.style.format('{:.0f}', na_rep='NaN')
                         .set_caption(f'{iso_i} -> {iso_j}: valores alternativos para años faltantes')
                         .set_properties(**{'font-size': '10px'}))

---
## Sección 6 — Nota metodológica: ¿qué hacer con los casos D?

Los corredores clasificados como **D) Sin imputación posible** son aquellos donde no hay ni fuente alternativa ni datos adyacentes para interpolar. Las opciones son:

1. **Excluir el corredor** del análisis de `vflow` (mantenerlo en `mflow` si tiene datos de Abel).

2. **Modelo de gravedad** — imputar usando un modelo Poisson estimado sobre los corredores con datos:
   ```
   log(vflow_ij_t) = alpha + beta1*log(GDP_i) + beta2*log(GDP_j)
                   + beta3*log(POP_i) + beta4*log(POP_j)
                   + beta5*log(distancia_ij) + FE_i + FE_j + FE_t
   ```
   Los valores predichos se usan como imputaciones. Es el método más defendible teóricamente para datos bilaterales.

3. **Compartir información entre corredores similares** — usar el ratio `vflow/mstock` de corredores parecidos (misma región, idioma) para inferir `vflow` a partir del `mstock` de Abel.

**Recomendación:** Para el DiD con PPML, la opción más conservadora es (1): excluir los corredores D del análisis de `vflow`, pero mantenerlos en `mflow`. Si se quiere imputar, la opción (2) es la más sólida metodológicamente.

> **Nota sobre redes neuronales:** Con 90 corredores × 28 años = 2520 observaciones, y patrones de missingness sistemáticos (corredores enteros sin dato), una red neuronal overfittearía. Una matrix factorization (SVD/NMF del panel corredor × año) o un modelo de gravedad son más apropiados y publicables.